# perinatal-ews: try it in your browser

**perinatal-ews** flags US counties where maternal and newborn outcomes (preterm birth, low birth weight, late or no prenatal care, gestational hypertension) are getting worse than expected, checks its own reliability, and forecasts next-year risk.

Run each cell in order (**Runtime → Run all** works too). Nothing is installed on your own computer.

- Code and documentation: https://github.com/karantejpal12/perinatal-ews
- Software DOI: https://doi.org/10.5281/zenodo.23381196
- Author: Karan Tejpal · ORCID 0009-0004-8609-2854

This is population surveillance to help health departments decide where to look first. It is not a clinical tool.

## 1. Install

In [ ]:
!pip install -q git+https://github.com/karantejpal12/perinatal-ews
import perinatal_ews
print('perinatal-ews', perinatal_ews.__version__)

## 2. Option A: the published 2024 results on real US data

The bulletin for 578 US counties (CDC WONDER natality data, 2016–2024), including the 2025 forecast, loaded from the project repository.

In [ ]:
import urllib.request
from IPython.display import HTML, display
base = 'https://raw.githubusercontent.com/karantejpal12/perinatal-ews/main/results/2024/'
html = urllib.request.urlopen(base + 'report.html').read().decode('utf-8')
display(HTML(html))

In [ ]:
import pandas as pd
detections = pd.read_csv(base + 'detections.csv', dtype={'geo_id': str})
alerts = detections[(detections.subgroup == 'All') & (detections.level == 'alert')]
alerts[['indicator', 'geo_name', 'state', 'births', 'rate', 'expected_rate', 'relative_risk']].head(20)

Look up one county (change the name and state):

In [ ]:
county, state = 'Hampden County', 'MA'
detections[(detections.geo_name == county) & (detections.state == state)][
    ['indicator', 'subgroup', 'level', 'rate', 'expected_rate', 'relative_risk', 'births']]

## 3. Option B: run the full pipeline on synthetic demo data

Generates fictional counties with known, injected deteriorations, runs detection, monitoring and the forecast, and shows the bulletin. Takes about a minute.

In [ ]:
!perinatal-ews demo --out demo
display(HTML(open('demo/report.html').read()))

## 4. Option C: run it on your own data

Upload a CSV in the perinatal-ews tidy format: one row per county, year, indicator and subgroup, with columns

`geo_id, geo_name, state, year, indicator, subgroup, numerator, denominator`

`subgroup` is `All` for the county total. Indicators must match the configuration (`preterm_birth`, `low_birth_weight`, `late_no_prenatal_care`, `gestational_hypertension`). You need at least 6 consecutive years; 8 or more is better. To get US county data, see the CDC WONDER instructions in the README.

Follow the CDC/NCHS data use rules: never publish counts of 9 or fewer.

In [ ]:
from google.colab import files
uploaded = files.upload()
name = next(iter(uploaded))
!perinatal-ews run --input "$name" --out my_results --forecast
display(HTML(open('my_results/report.html').read()))

In [ ]:
# Download all result files as a zip
!cd my_results && zip -q -r ../my_results.zip .
files.download('my_results.zip')